# Example 3: Binary Classification on Heart Disease Dataset using SONFIS

In [1]:
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report
)

import torch
import torch.nn as nn
import torch.utils.data as data

import numpy as np
import random

import neuro_fuzzy_toolbox as nft

In [2]:
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Data

In [3]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

# Fill missing values
X = X.fillna(value=0)

# Convert to binary classification: 0 = no disease, 1 = disease
y = y.copy()
y.loc[y['num'] > 0, 'num'] = 1

In [4]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)

x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.2, stratify=y_train, random_state=SEED
)

In [5]:
scaler = MinMaxScaler(feature_range=(0, 1))

x_train = torch.from_numpy(scaler.fit_transform(x_train)).to(torch.float64)
x_val   = torch.from_numpy(scaler.transform(x_val)).to(torch.float64)
x_test  = torch.from_numpy(scaler.transform(x_test)).to(torch.float64)

y_train = torch.from_numpy(y_train.values).squeeze()
y_val   = torch.from_numpy(y_val.values).squeeze()
y_test  = torch.from_numpy(y_test.values).squeeze()

## DataLoaders

In [6]:
generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = data.DataLoader(
    data.TensorDataset(x_train, y_train),
    batch_size=16, shuffle=True, generator=generator
)

val_loader = data.DataLoader(
    data.TensorDataset(x_val, y_val),
    batch_size=16, shuffle=False
)

## Model

In [7]:
features = heart_disease.variables["name"][:13].tolist()

# Define model
model = nft.rule_reduced_ANFIS(
    input_size=x_train.shape[1],
    num_mfs=3,
    outputs=2,
    membership_function=nft.Gaussian_MF(),
    output_type='softmax',
    features=features,
    dtype=torch.float64
)

In [8]:
model.init_premises(x_train)
model.init_consequents(x_train, y_train, driver="gelsd", ridge_lambda=1e-3)

## Learning Algorithm

In [9]:
# Define the parameter training algorithm to be used inside SONFIS
anfis_trainer = nft.Basic_optimizer_training_algorithm(
    epochs=1000,
    loss_function=nn.CrossEntropyLoss(),
    optimizer=torch.optim.AdamW,
    optimizer_params={'lr': 1e-3, 'weight_decay': 1e-2},
    early_stopping=nft.EarlyStopping(patience=80)
)

## SONFIS

In [10]:
# Define SONFIS
sonfis = nft.SONFIS(
    Ngrow=20,
    dGrow=0.8,
    Nsplit=25,
    eSplit=0.35,
    Nvanish=5,
    lVanish=4,
    max_iterations=100,
    ANFIStrainer=anfis_trainer,
    early_stopping=nft.EarlyStopping(patience=25),
    lse_for_new_consequents=True,
    lse_for_new_consequents_lambda=1e-1,
    last_training_iteration=False
)

In [11]:
# Train model
sonfis(model, train_loader, val_loader)

ITERATION:   0/100


/home/jsuarez/workspaces/neuro-fuzzy-toolbox/env/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass



STARTING STATE:
   premises                                                                                                                                                                                                                                                           output 1 consequents                                                                                                                                   output 2 consequents                                                                                                                                 
        age                 sex                  cp            trestbps                chol                 fbs             restecg             thalach               exang             oldpeak               slope                  ca                thal                            age       sex        cp  trestbps      chol       fbs   restecg   thalach     exang   oldpeak     slope        ca      thal                  

## Evaluation

In [12]:
pred = model.predict(x_test)

acc        = accuracy_score(y_test, pred)
prec       = precision_score(y_test, pred, zero_division=0)
recall     = recall_score(y_test, pred)
f1         = f1_score(y_test, pred, zero_division=0)
conf_matrix = confusion_matrix(y_test, pred)
class_rep  = classification_report(y_test, pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("F1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

print("Classification Report:")
print(class_rep)

Accuracy: 0.8131868131868132
Precision: 0.8378378378378378
Recall: 0.7380952380952381
F1 score: 0.7848101265822784 

Confusion Matrix:
[[43  6]
 [11 31]] 

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.88      0.83        49
           1       0.84      0.74      0.78        42

    accuracy                           0.81        91
   macro avg       0.82      0.81      0.81        91
weighted avg       0.82      0.81      0.81        91



In [13]:
print(model.rules)

6
